# mT5-Gold RAG Answer Evaluation (Validation)

## Purpose

This notebook evaluates the answers generated by `07a_mt5_gold_rag.ipynb` on the **validation** split, for each of the three retrievers (TF-IDF, BM25, dense). 

## Metrics

Same three metrics as the Qwen evaluation, for direct comparability:

- **Exact Match (EM)**
- **Token F1** (SQuAD-style)
- **ROUGE-L F1** (LCS-based)

In [18]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import altair as alt

In [19]:
GENERATOR_NAME = "mt5_gold"

RETRIEVERS = ("tfidf", "bm25", "dense")

RAG_TOP_K = 5


SPLIT = "validation"

PROJECT_ROOT = Path.cwd()

GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "generation"
    / "mt5_gold"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "mt5_gold"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert GENERATION_DIR.exists(), (
    f"Ne postoji: {GENERATION_DIR}. "
    "Prekopiraj *_predictions.jsonl fajlove iz "
    "07a_mt5_gold_rag.ipynb (ARTIFACTS_DIR/generation/) ovde."
)

print("Generisane predikcije:", GENERATION_DIR)
print("Rezultati evaluacije:", RESULTS_DIR)
print("Split koji se evaluira:", SPLIT)
print("RAG_TOP_K:", RAG_TOP_K)

Generisane predikcije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/generation/mt5_gold
Rezultati evaluacije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/mt5_gold
Split koji se evaluira: validation
RAG_TOP_K: 5


In [20]:
def load_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

## Loading Generated Predictions

In [21]:
required_fields = {
    "question_id",
    "question",
    "gold_answer",
    "retriever",
    "generator",
    "generated_answer",
}

generation_data = {}

for retriever in RETRIEVERS:
    path = GENERATION_DIR / f"{retriever}_{SPLIT}_top{RAG_TOP_K}_predictions.jsonl"

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    for record in records:
        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

    generation_data[retriever] = records
    print(f"{retriever}/{SPLIT}: {len(records)} generisanih odgovora")

if not generation_data:
    raise FileNotFoundError(
        f"Nisu pronađeni generisani odgovori za split '{SPLIT}' ni za jedan retriever."
    )

tfidf/validation: 21 generisanih odgovora
bm25/validation: 21 generisanih odgovora
dense/validation: 21 generisanih odgovora


## Text Normalization and Metrics

In [22]:
TOKEN_PATTERN = re.compile(r"[\w]+", re.UNICODE)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str):
    return TOKEN_PATTERN.findall(normalize_text(text))

In [23]:
def compute_exact_match(prediction: str, reference: str) -> int:
    return int(normalize_text(prediction) == normalize_text(reference))


def compute_token_f1(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    from collections import Counter

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    overlap = sum(
        min(pred_counts[token], ref_counts[token])
        for token in pred_counts
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [24]:
def longest_common_subsequence_length(a: list, b: list) -> int:
    previous_row = [0] * (len(b) + 1)

    for token_a in a:
        current_row = [0] * (len(b) + 1)

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current_row[j] = previous_row[j - 1] + 1
            else:
                current_row[j] = max(previous_row[j], current_row[j - 1])

        previous_row = current_row

    return previous_row[-1]


def compute_rouge_l(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(pred_tokens, ref_tokens)

    if lcs_length == 0:
        return 0.0

    precision = lcs_length / len(pred_tokens)
    recall = lcs_length / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

## Per-Question and Aggregate Metrics

In [25]:
def evaluate_generations(records: list, retriever: str) -> pd.DataFrame:
    rows = []

    for record in records:
        prediction = record["generated_answer"]
        reference = record["gold_answer"]

        rows.append({
            "question_id": record["question_id"],
            "retriever": retriever,
            "EM": compute_exact_match(prediction, reference),
            "F1": compute_token_f1(prediction, reference),
            "ROUGE_L": compute_rouge_l(prediction, reference),
        })

    return pd.DataFrame(rows)


per_question_frames = [
    evaluate_generations(records, retriever)
    for retriever, records in generation_data.items()
]

per_question_df = pd.concat(per_question_frames, ignore_index=True)

per_question_df.head()

,question_id,retriever,EM,F1,ROUGE_L
0,132,tfidf,0,0.064516,0.064516
1,125,tfidf,0,0.071429,0.071429
2,84,tfidf,0,0.129032,0.129032
3,141,tfidf,0,0.066667,0.066667
4,114,tfidf,0,0.130435,0.086957


In [26]:
metrics_df = (
    per_question_df
    .groupby("retriever")[["EM", "F1", "ROUGE_L"]]
    .mean()
    .reset_index()
    .sort_values("F1", ascending=False)
)

metrics_df

,retriever,EM,F1,ROUGE_L
2,tfidf,0.0,0.091717,0.073055
1,dense,0.0,0.079957,0.074005
0,bm25,0.0,0.078872,0.068615


## Comparing Retrievers on Validation

In [27]:
metrics_plot_df = metrics_df.melt(
    id_vars="retriever",
    value_vars=["EM", "F1", "ROUGE_L"],
    var_name="metric",
    value_name="score",
)

alt.Chart(metrics_plot_df).mark_bar().encode(
    x=alt.X("retriever:N", title="Retriever"),
    y=alt.Y("score:Q", title="Score", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("metric:N", title="Metrika"),
    xOffset="metric:N",
    tooltip=["retriever", "metric", alt.Tooltip("score:Q", format=".3f")],
).properties(
    title=f"mT5-Gold — poređenje retrievera na {SPLIT} skupu",
    width=500,
    height=350,
)

alt.Chart(...)

## Inspecting Individual Answers

In [28]:
best_retriever = metrics_df.iloc[0]["retriever"]

sample_df = (
    per_question_df[per_question_df["retriever"] == best_retriever]
    .sort_values("F1")
    .head(5)
)

sample_records = {
    record["question_id"]: record
    for record in generation_data[best_retriever]
}

for _, row in sample_df.iterrows():
    record = sample_records[row["question_id"]]
    print(f"Pitanje: {record['question']}")
    print(f"Referentni odgovor: {record['gold_answer']}")
    print(f"Generisani odgovor: {record['generated_answer']}")
    print(f"EM={row['EM']:.0f}  F1={row['F1']:.3f}  ROUGE_L={row['ROUGE_L']:.3f}")
    print("-" * 80)

Pitanje: Opisati testove sigurnosti pri testiranju softvera.
Referentni odgovor: Testovi sigurnosti proveravaju zaštitu funkcionalnosti i podataka od neovlašćenih korisnika, uključujući dostupnost, integritet i poverljivost podataka.
Generisani odgovor: <extra_id_0> u odnosu na uslov. Testiranje sive kutije
EM=0  F1=0.000  ROUGE_L=0.000
--------------------------------------------------------------------------------
Pitanje: Definisati metamorfno testiranje pri testiranju softvera.
Referentni odgovor: Metamorfno testiranje ne zahteva poznavanje konkretnog očekivanog izlaza. Umesto toga proverava odnose između izlaza dobijenih za povezane ulaze i koristi poznate osobine sistema. Značajno doprinosi povećanju stepena automatizacije testiranja i otkrivanju grešaka u sistemima sa problemom proročišta.
Generisani odgovor: <extra_id_0> u odnosu na odnos. Odgovori na pitanje:
EM=0  F1=0.044  ROUGE_L=0.044
--------------------------------------------------------------------------------
Pitanje:

In [ ]:
per_question_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_per_question_metrics.csv",
    index=False,
)

metrics_df_with_generator = metrics_df.copy()
metrics_df_with_generator.insert(0, "generator", GENERATOR_NAME)

metrics_df_with_generator.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metrics.csv",
    index=False,
)

metadata = {
    "generator": GENERATOR_NAME,
    "split": SPLIT,
    "rag_top_k": RAG_TOP_K,
    "retrievers_evaluated": list(generation_data.keys()),
    "metrics": ["EM", "F1", "ROUGE_L"],
    "n_questions_per_retriever": {
        retriever: len(records)
        for retriever, records in generation_data.items()
    },
}

with (RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Sačuvani rezultati evaluacije u:", RESULTS_DIR)